# 01 - Carregamento e Limpeza

Este notebook mostra, de forma did?tica, como a Aurora Finance Intelligence parte do dataset p?blico **Churn Modelling** do Kaggle e chega ao schema interno usado no projeto.

A ideia aqui n?o ? substituir o pipeline oficial em `src/`, mas deixar vis?vel para a banca o racioc?nio de carregamento, inspe??o, limpeza e padroniza??o.

Antes de executar este notebook, rode na raiz do projeto:

```powershell
python -m src.pipeline
```

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DADOS_RAW = ROOT / "dados" / "raw"
DADOS_PROCESSED = ROOT / "dados" / "processed"
DADOS_OUTPUTS = ROOT / "dados" / "outputs"
print(f"Raiz do projeto: {ROOT}")

## 1. Dataset p?blico esperado

O arquivo p?blico deve estar em:

`dados/raw/churn_modelling.csv`

O projeto **n?o baixa automaticamente do Kaggle**, porque o Kaggle exige autentica??o. Se o arquivo n?o existir, o pipeline usa fallback sint?tico e continua funcionando.

In [ ]:
raw_path = DADOS_RAW / "churn_modelling.csv"
raw_path.exists(), raw_path

In [ ]:
if raw_path.exists():
    raw = pd.read_csv(raw_path)
    print(f"Linhas: {raw.shape[0]:,} | Colunas: {raw.shape[1]:,}")
    display(raw.head())
else:
    raw = pd.DataFrame()
    print("Arquivo p?blico n?o encontrado. O pipeline oficial usar? fallback sint?tico.")

## 2. Colunas originais do Churn Modelling

Quando o CSV p?blico existe, as colunas esperadas s?o:

`RowNumber`, `CustomerId`, `Surname`, `CreditScore`, `Geography`, `Gender`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`, `EstimatedSalary`, `Exited`.

In [ ]:
if not raw.empty:
    display(pd.DataFrame({"coluna_original": raw.columns, "tipo": raw.dtypes.astype(str).values}))

## 3. Diagn?stico inicial

Aqui verificamos nulos, duplicatas e tipos. Isso ajuda a documentar a qualidade da fonte antes da padroniza??o.

In [ ]:
if not raw.empty:
    diagnostico = pd.DataFrame({
        "nulos": raw.isna().sum(),
        "pct_nulos": (raw.isna().mean() * 100).round(2),
        "tipo": raw.dtypes.astype(str),
        "valores_unicos": raw.nunique(dropna=True),
    })
    display(diagnostico)
    print(f"Duplicatas completas: {raw.duplicated().sum():,}")
    if "CustomerId" in raw.columns:
        print(f"CustomerId duplicados: {raw['CustomerId'].duplicated().sum():,}")

## 4. Convers?es para o schema Aurora

Principais regras usadas no projeto:

- `CustomerId` vira `customer_id_original` e a Aurora cria `cliente_id` est?vel no formato `C000001`.
- `Surname` vira `nome`.
- `Age` vira `idade`.
- `Gender` vira `genero`, padronizado para `Feminino`, `Masculino` ou `Nao informado`.
- `Geography` vira `estado` e tamb?m ajuda a criar `cidade`.
- `EstimatedSalary` ? tratado como sal?rio anual estimado e convertido para `renda_mensal` dividindo por 12.
- `Balance` vira `saldo_atual`.
- `CreditScore` vira `score_credito`.
- `Tenure` vira `tempo_relacionamento`.
- `NumOfProducts` vira `produtos_ativos`.
- `HasCrCard` vira `tem_cartao_credito`.
- `IsActiveMember` vira `membro_ativo`.
- `Exited` vira `churn_flag`, o target oficial do problema.

In [ ]:
def traduz_genero(valor):
    texto = str(valor).strip().lower()
    if texto in ["female", "feminino", "f"]:
        return "Feminino"
    if texto in ["male", "masculino", "m"]:
        return "Masculino"
    return "Nao informado"

mapa_estado = {"France": "Franca", "Spain": "Espanha", "Germany": "Alemanha"}
mapa_cidade = {"Franca": "Paris", "Espanha": "Madrid", "Alemanha": "Berlim"}

if not raw.empty:
    exemplo = pd.DataFrame({
        "cliente_id": [f"C{i:06d}" for i in range(1, len(raw) + 1)],
        "customer_id_original": raw["CustomerId"],
        "nome": raw["Surname"].fillna("Cliente"),
        "idade": pd.to_numeric(raw["Age"], errors="coerce"),
        "genero": raw["Gender"].map(traduz_genero),
        "estado": raw["Geography"].map(mapa_estado).fillna(raw["Geography"].astype(str)),
        "renda_mensal": pd.to_numeric(raw["EstimatedSalary"], errors="coerce") / 12,
        "saldo_atual": pd.to_numeric(raw["Balance"], errors="coerce"),
        "score_credito": pd.to_numeric(raw["CreditScore"], errors="coerce"),
        "tempo_relacionamento": pd.to_numeric(raw["Tenure"], errors="coerce"),
        "produtos_ativos": pd.to_numeric(raw["NumOfProducts"], errors="coerce"),
        "tem_cartao_credito": pd.to_numeric(raw["HasCrCard"], errors="coerce"),
        "membro_ativo": pd.to_numeric(raw["IsActiveMember"], errors="coerce"),
        "churn_flag": pd.to_numeric(raw["Exited"], errors="coerce").fillna(0).astype(int),
    })
    exemplo["cidade"] = exemplo["estado"].map(mapa_cidade).fillna(exemplo["estado"])
    exemplo["origem_dado"] = "public_kaggle_churn_modelling"
    display(exemplo.head())

## 5. Dados processados pelo pipeline oficial

A partir daqui, lemos os CSVs j? gerados por `python -m src.pipeline`. Essa ? a vers?o que alimenta SQL, Power BI, modelo e app React.

In [ ]:
clientes = pd.read_csv(DADOS_PROCESSED / "clientes_limpo.csv")
transacoes = pd.read_csv(DADOS_PROCESSED / "transacoes_limpo.csv")
categorias = pd.read_csv(DADOS_PROCESSED / "categorias_limpo.csv")

print("Clientes:", clientes.shape)
print("Transa??es:", transacoes.shape)
print("Categorias:", categorias.shape)
display(clientes.head())

In [ ]:
resumo_processado = pd.DataFrame({
    "tabela": ["clientes_limpo", "transacoes_limpo", "categorias_limpo"],
    "linhas": [len(clientes), len(transacoes), len(categorias)],
    "colunas": [clientes.shape[1], transacoes.shape[1], categorias.shape[1]],
    "nulos_total": [clientes.isna().sum().sum(), transacoes.isna().sum().sum(), categorias.isna().sum().sum()],
    "duplicatas": [clientes.duplicated().sum(), transacoes.duplicated().sum(), categorias.duplicated().sum()],
})
display(resumo_processado)

## 6. Observa??o importante sobre as transa??es

O `Churn Modelling` traz dados de perfil e churn, mas **n?o possui hist?rico transacional por categoria**. Por isso, a Aurora cria uma camada chamada **Aurora Synthetic Transactions Layer**.

Essa camada ? sint?tica, reprodut?vel e serve para demonstrar an?lise explorat?ria, SQL, Power BI e storytelling de consumo. Ela n?o altera o target real de churn quando o CSV p?blico existe.

In [ ]:
print("Origem dos clientes:")
display(clientes["origem_dado"].value_counts().rename_axis("origem_dado").reset_index(name="qtd_clientes"))

print("Per?odo transacional gerado:")
display(transacoes["data"].agg(["min", "max"]))